# Assembly Line Model Comparison Notebook

This notebook is a template for benchmarking multiple ML models across all five datasets:

- base dataset: `dataset.csv`
- sparse variant: `dataset_variant_sparse.csv`
- drift variant: `dataset_variant_drift.csv`
- propagation variant: `dataset_variant_propagation.csv`
- noisy variant: `dataset_variant_noisy.csv`

It is designed for comparing different supervised models and selecting the best one for robustness.

The notebook includes:
- dataset loading
- preprocessing
- model training and validation
- metric comparison
- ranking of models across variants

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

import lightgbm as lgb

# Optional xgboost if installed
try:
    import xgboost as xgb
except Exception:
    xgb = None

print("Libraries imported")

In [ ]:
DATA_DIR = Path.cwd()
DATASETS = {
    "base": DATA_DIR / "dataset.csv",
    "sparse": DATA_DIR / "dataset_variant_sparse.csv",
    "drift": DATA_DIR / "dataset_variant_drift.csv",
    "propagation": DATA_DIR / "dataset_variant_propagation.csv",
    "noisy": DATA_DIR / "dataset_variant_noisy.csv",
}

for name, path in DATASETS.items():
    print(name, path.exists(), path)

In [ ]:
# ------------------------------------------------------------------------------
# 1) Define your model factory functions here.
# ------------------------------------------------------------------------------
# You only need to write model creation code here.
# The rest of the notebook will handle preprocessing, dataset loops, training, metrics, and printing.

TARGET = "anomaly_flag"
DROP_COLS = ["vehicle_id", "timestamp", "station_name", "root_cause_station"]
NUMERIC_COLS = [
    "cycle_time_sec",
    "torque_nm",
    "temperature_c",
    "vibration_rms",
    "pressure_bar",
    "force_n",
    "position_error_mm",
    "voltage_v",
    "current_a",
    "flow_rate_lpm",
    "queue_time_sec",
    "ambient_temperature_c",
    "humidity_pct",
]
CATEGORICAL_COLS = [
    "vehicle_model",
    "vehicle_variant",
    "station_id",
    "shift",
    "production_batch",
]


def create_logistic_regression(random_state=42):
    return LogisticRegression(max_iter=5000, class_weight="balanced", random_state=random_state)


def create_random_forest(random_state=42):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        random_state=random_state,
    )


def create_extra_trees(random_state=42):
    return ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=random_state,
    )


def create_decision_tree(random_state=42):
    return DecisionTreeClassifier(class_weight="balanced", random_state=random_state)


def create_svm(random_state=42):
    return SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=random_state)


def create_knn(random_state=42):
    return KNeighborsClassifier(n_neighbors=10)


def create_naive_bayes(random_state=42):
    return GaussianNB()


def create_histgbm(random_state=42):
    return HistGradientBoostingClassifier(random_state=random_state)


def create_lightgbm(random_state=42):
    return lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        learning_rate=0.05,
        n_estimators=400,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=random_state,
        verbosity=-1,
    )


def create_xgboost(random_state=42):
    if xgb is None:
        raise ValueError("xgboost is not installed")
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=random_state,
    )


# ------------------------------------------------------------------------------
# 2) Add your own model here.
# ------------------------------------------------------------------------------
# Example:
# def create_my_model(random_state=42):
#     from sklearn.ensemble import GradientBoostingClassifier
#     return GradientBoostingClassifier(random_state=random_state)
#
# Then add "my_model" to MODEL_NAMES below.

MODEL_DEFINITIONS = {
    "logistic_regression": create_logistic_regression,
    "random_forest": create_random_forest,
    "extra_trees": create_extra_trees,
    "decision_tree": create_decision_tree,
    "svm": create_svm,
    "knn": create_knn,
    "naive_bayes": create_naive_bayes,
    "histgbm": create_histgbm,
    "lightgbm": create_lightgbm,
}

if xgb is not None:
    MODEL_DEFINITIONS["xgboost"] = create_xgboost


def build_model(model_name, random_state=42):
    if model_name not in MODEL_DEFINITIONS:
        raise ValueError(f"Model '{model_name}' is not registered. Add it to MODEL_DEFINITIONS.")
    return MODEL_DEFINITIONS[model_name](random_state=random_state)


def load_dataset(path):
    df = pd.read_csv(path)
    df = df.copy()
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def prepare_features(df):
    X = df.drop(columns=[TARGET] + DROP_COLS, errors="ignore")
    y = df[TARGET].astype(int)
    return X, y


def make_preprocessor():
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERIC_COLS),
            ("cat", categorical_transformer, CATEGORICAL_COLS),
        ]
    )
    return preprocessor


def evaluate_model(model_name, X_train, X_test, y_train, y_test):
    model = build_model(model_name)
    preprocessor = make_preprocessor()

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }

    return model, pipe, metrics, y_pred, y_prob


# ------------------------------------------------------------------------------
# 3) Choose which models to run.
# ------------------------------------------------------------------------------
# Add your own model key here, for example:
# MODEL_NAMES = ["lightgbm", "random_forest", "my_model"]

MODEL_NAMES = [
    "logistic_regression",
    "random_forest",
    "extra_trees",
    "decision_tree",
    "svm",
    "knn",
    "naive_bayes",
    "histgbm",
    "lightgbm",
]

if xgb is not None:
    MODEL_NAMES.append("xgboost")

print("Registered models:", list(MODEL_DEFINITIONS.keys()))
print("Models to evaluate:", MODEL_NAMES)

In [ ]:
results = []

for dataset_name, dataset_path in DATASETS.items():
    df = load_dataset(dataset_path)
    X, y = prepare_features(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    for model_name in MODEL_NAMES:
        model, pipe, metrics, y_pred, y_prob = evaluate_model(model_name, X_train, X_test, y_train, y_test)

        results.append({
            "dataset": dataset_name,
            "model": model_name,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"],
        })

        print(f"{dataset_name:>12} | {model_name:>18} | AUC={metrics['roc_auc']:.4f} | F1={metrics['f1']:.4f}")

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.sort_values(["dataset", "roc_auc"], ascending=[True, False]))

In [ ]:
# Create a model rank summary by dataset
rank_summary = []
for dataset_name, group in results_df.groupby("dataset"):
    ranked = group.sort_values("roc_auc", ascending=False).reset_index(drop=True)
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    rank_summary.append(ranked)

rank_summary_df = pd.concat(rank_summary, ignore_index=True)
print(rank_summary_df[["dataset", "model", "rank", "roc_auc", "f1"]].sort_values(["dataset", "rank"]))

In [ ]:
# Choose the best overall model across all datasets by average AUC
model_score = (
    results_df.groupby("model")["roc_auc"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("Best overall model by average AUC across variants:")
print(model_score)

## Notes

- Replace `MODEL_NAMES` with any subset of models you want to test.
- Add more models as needed.
- Add cross-validation if you want more stable estimates.
- Keep the target as `anomaly_flag` for the anomaly detection benchmark.
- For root-cause modeling, repeat the same structure with `root_cause_station` as the target.

In [ ]:
# Optional: save results to file
result_path = DATA_DIR / "model_comparison_results.csv"
results_df.to_csv(result_path, index=False)
print(f"Saved results to: {result_path}")